In [1]:
# Cell 1: Imports for the entire RAG pipeline (extraction → chunking → embedding → storage → retrieval → generation)

import os
from pathlib import Path
from dotenv import load_dotenv

from pypdf import PdfReader          # PDF text extraction
import chromadb                       # vector database
from google import genai              # Gemini API (used for both embeddings and final generation)

# Load environment variables from backend/.env so we can reuse the same GEMINI_API_KEY
env_path = Path(r"C:\Users\T Ganesh\Desktop\gsg\Gadsik.ai\backend\.env")
load_dotenv(dotenv_path=env_path)

GEMINI_API_KEY = os.environ["GEMINI_API_KEY"]

client = genai.Client(api_key=GEMINI_API_KEY)

print("All imports loaded successfully.")
print(f"Gemini client ready: {client is not None}")

All imports loaded successfully.
Gemini client ready: True


In [2]:
# Cell 2: Production PDF Reader (Page-level text extraction)
pdf_path = r"C:\Users\T Ganesh\Desktop\gsg\Gadsik.ai\data\Introduction to Machine Learning with Python ( PDFDrive.com )-min.pdf"

def extract_text_from_pdf(pdf_path: str) -> list[dict]:
    """
    Extracts text page-by-page from a PDF document.
    Returns a list of dicts with page number and cleaned text.
    """
    path = Path(pdf_path)
    if not path.exists():
        raise FileNotFoundError(f"PDF file not found at: {pdf_path}")
        
    reader = PdfReader(str(path))
    extracted_pages = []
    
    for page_idx, page in enumerate(reader.pages):
        raw_text = page.extract_text() or ""
        # Clean basic whitespace artifacts
        cleaned_text = " ".join(raw_text.split())
        
        if cleaned_text:
            extracted_pages.append({
                "page_number": page_idx + 1,
                "text": cleaned_text
            })
            
    print(f"✓ Extracted {len(extracted_pages)} non-empty pages from '{path.name}'.")
    return extracted_pages

# Run extraction and store in variable for Cell 3
pdf_pages = extract_text_from_pdf(pdf_path)


✓ Extracted 387 non-empty pages from 'Introduction to Machine Learning with Python ( PDFDrive.com )-min.pdf'.


In [3]:
print(pdf_pages[0])  # first page's dict
print(f"\nTotal pages with content: {len(pdf_pages)}")

{'page_number': 1, 'text': 'Andreas C. Müller & Sarah Guido Introduction to Machine Learning with P y thon A GUIDE FOR DATA SCIENTISTS'}

Total pages with content: 387


In [4]:
# Cell 3: Chunk the extracted page text into smaller retrievable pieces

def chunk_text(pdf_pages: list[dict], chunk_size: int = 500, overlap: int = 50) -> list[dict]:
    """
    Splits page-level text into overlapping chunks.
    Each chunk retains a reference to its source page number.
    """
    chunks = []
    chunk_id = 0

    for page in pdf_pages:
        text = page["text"]
        page_number = page["page_number"]

        start = 0
        while start < len(text):
            end = start + chunk_size
            chunk_text_piece = text[start:end]

            chunks.append({
                "chunk_id": chunk_id,
                "page_number": page_number,
                "text": chunk_text_piece
            })
            chunk_id += 1

            start += chunk_size - overlap  # move forward, keeping overlap

    print(f"✓ Created {len(chunks)} chunks from {len(pdf_pages)} pages.")
    return chunks

# Run chunking
doc_chunks = chunk_text(pdf_pages)

# Inspect a sample chunk
print("\n--- Sample chunk (index 5) ---")
print(doc_chunks[5])

✓ Created 1682 chunks from 387 pages.

--- Sample chunk (index 5) ---
{'chunk_id': 5, 'page_number': 4, 'text': 'and the authors disclaim all responsibility for errors or omissions, including without limitation responsibility for damages resulting from the use of or reliance on this work. Use of the information and instructions contained in this work is at your own risk. If any code samples or other technology this work contains or describes is subject to open source licenses or the intellectual property rights of others, it is your responsibility to ensure that your use thereof complies with such licenses'}


In [5]:
# Cell 4 (revised): Use a local, free embedding model instead of Gemini's paid API

from sentence_transformers import SentenceTransformer

# This downloads a small, fast, good-quality embedding model the first time you run it
# (runs entirely on your machine after that - no API calls, no cost)
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

def get_embedding(text: str) -> list[float]:
    """
    Generates a vector embedding locally - free, no API quota used.
    """
    vector = embedding_model.encode(text)
    return vector.tolist()

# Test on one chunk first
sample_embedding = get_embedding(doc_chunks[5]["text"])
print(f"Embedding vector length: {len(sample_embedding)}")
print(f"First 10 values: {sample_embedding[:10]}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding vector length: 384
First 10 values: [-0.13212285935878754, 0.10656233131885529, -0.05193266272544861, -0.04847964271903038, 0.029020221903920174, -0.08438342809677124, -0.028697984293103218, 0.0058095273561775684, -0.019421085715293884, 0.005513963755220175]


In [7]:
import chromadb

# 1. Initialize the ChromaDB client first
chroma_client = chromadb.PersistentClient(path=r"C:\Users\T Ganesh\Desktop\gsg\Gadsik.ai\notebook\chroma_db")

# 2. Safely try to delete the old collection if it exists
try:
    chroma_client.delete_collection(name="ml_python_book")
    print("Old collection deleted successfully.")
except Exception as e:
    print("Collection did not exist yet, starting fresh.")

# 3. Create a fresh collection for 384-dimensional vectors
collection = chroma_client.get_or_create_collection(name="ml_python_book")

print("✓ Collection reset and ready!")


Old collection deleted successfully.
✓ Collection reset and ready!


In [8]:
# Delete the old collection (created with Gemini's 3072-dim embeddings) and start clean
chroma_client.delete_collection(name="ml_python_book")

# Recreate it fresh - will now correctly expect 384-dim vectors
collection = chroma_client.get_or_create_collection(name="ml_python_book")

print("Collection reset. Ready for 384-dim embeddings.")

Collection reset. Ready for 384-dim embeddings.


In [9]:
texts = [chunk["text"] for chunk in doc_chunks]
ids = [str(chunk["chunk_id"]) for chunk in doc_chunks]
metadatas = [{"page_number": chunk["page_number"]} for chunk in doc_chunks]

print(f"Embedding {len(texts)} chunks locally...")
embeddings = embedding_model.encode(texts, show_progress_bar=True).tolist()

print("Storing in ChromaDB...")
collection.add(
    ids=ids,
    embeddings=embeddings,
    documents=texts,
    metadatas=metadatas
)

print(f"\n✓ Done. Collection now has {collection.count()} chunks stored.")

Embedding 1682 chunks locally...


Batches:   0%|          | 0/53 [00:00<?, ?it/s]

Storing in ChromaDB...

✓ Done. Collection now has 1682 chunks stored.


In [10]:
# Cell 6: Query the collection - retrieve the most relevant chunks for a question

def retrieve_relevant_chunks(question: str, top_k: int = 5) -> list[dict]:
    """
    Embeds the question and finds the top_k most similar chunks in ChromaDB.
    """
    question_embedding = embedding_model.encode(question).tolist()

    results = collection.query(
        query_embeddings=[question_embedding],
        n_results=top_k
    )

    retrieved = []
    for i in range(len(results["ids"][0])):
        retrieved.append({
            "chunk_id": results["ids"][0][i],
            "text": results["documents"][0][i],
            "page_number": results["metadatas"][0][i]["page_number"],
            "distance": results["distances"][0][i]
        })
    return retrieved

# Test with a real question about the book's content
question = "What is supervised learning?"
results = retrieve_relevant_chunks(question)

print(f"Question: {question}\n")
for r in results:
    print(f"[Page {r['page_number']}, distance {r['distance']:.4f}]")
    print(r["text"][:200])
    print("---")

Question: What is supervised learning?

[Page 50, distance 0.5545]
ussed in Chapter 1: 36 | Chapter 2: Supervised Learning
---
[Page 39, distance 0.7406]
CHAPTER 2 Supervised Learning As we mentioned earlier, supervised machine learning is one of the most commonly used and successful types of machine learning. In this chapter, we will describe super‐ v
---
[Page 39, distance 0.7501]
wers. Remember that supervised learning is used whenever we want to predict a certain outcome from a given input, and we have examples of input/output pairs. We build a machine learning model from the
---
[Page 132, distance 0.7506]
 with many additional param‐ 118 | Chapter 2: Supervised Learning
---
[Page 16, distance 0.8356]
ning algorithms that learn from input/output pairs are called supervised learning algorithms because a “teacher” provides supervision to the algorithms in the form of the desired outputs for each exam
---


In [11]:
# Cell 7: Generate a final answer using retrieved chunks as context (this is the "RAG" part)

def answer_question(question: str, top_k: int = 5) -> str:
    """
    Full RAG: retrieve relevant chunks, inject into prompt, generate answer.
    """
    retrieved_chunks = retrieve_relevant_chunks(question, top_k=top_k)

    # Build context from retrieved chunks
    context = "\n\n".join(
        f"[Page {c['page_number']}]: {c['text']}" for c in retrieved_chunks
    )

    prompt = f"""You are answering a question using ONLY the context provided below, 
which comes from a machine learning textbook. If the context doesn't contain 
enough information to answer, say so honestly.

Context:
{context}

Question: {question}

Answer:"""

    response = client.models.generate_content(
        model="gemini-flash-latest",
        contents=prompt,
    )
    return response.text

# Test it
question = "What is supervised learning?"
answer = answer_question(question)

print(f"Question: {question}\n")
print(f"Answer:\n{answer}")

Question: What is supervised learning?

Answer:
Based on the provided context, supervised learning is a type of machine learning where algorithms learn from examples of input/output pairs (which make up a training set) to predict a certain outcome from a given input. 

Key aspects mentioned in the context include:
* **Supervision:** It is called "supervised" because a "teacher" provides supervision to the algorithms in the form of desired outputs for each example they learn from.
* **Goal:** The goal is to build a machine learning model using these input/output pairs to make accurate predictions for new, never-before-seen data.
* **Process:** Creating the training set often requires manual human effort, but the model can afterward automate and speed up tasks that would otherwise be laborious or infeasible.


In [13]:
import sys
sys.path.append(r"C:\Users\T Ganesh\Desktop\gsg\Gadsik.ai\backend")

from app.rag.chunking import extract_text_from_pdf, chunk_text

pages = extract_text_from_pdf(pdf_path)
chunks = chunk_text(pages)

print(f"✓ Pages: {len(pages)}, Chunks: {len(chunks)}")
print(chunks[5])

✓ Pages: 387, Chunks: 1682
{'chunk_id': 5, 'page_number': 4, 'text': 'and the authors disclaim all responsibility for errors or omissions, including without limitation responsibility for damages resulting from the use of or reliance on this work. Use of the information and instructions contained in this work is at your own risk. If any code samples or other technology this work contains or describes is subject to open source licenses or the intellectual property rights of others, it is your responsibility to ensure that your use thereof complies with such licenses'}


In [14]:
from app.rag.embeddings import embed_texts, embed_query

sample_vec = embed_query("What is supervised learning?")
print(f"Query embedding length: {len(sample_vec)}")

batch_vecs = embed_texts([chunks[0]["text"], chunks[1]["text"]])
print(f"Batch embeddings: {len(batch_vecs)} vectors, each length {len(batch_vecs[0])}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Query embedding length: 384
Batch embeddings: 2 vectors, each length 384


In [15]:
from app.rag.chroma_store import store_chunks, query_collection
from app.rag.embeddings import embed_texts, embed_query

# Use a test collection name so we don't touch your original "ml_python_book" collection
test_embeddings = embed_texts([c["text"] for c in chunks[:20]])
count = store_chunks("test_collection", chunks[:20], test_embeddings)
print(f"Stored {count} chunks in test collection")

query_vec = embed_query("What is supervised learning?")
results = query_collection("test_collection", query_vec, top_k=3)
for r in results:
    print(f"[Page {r['page_number']}, distance {r['distance']:.4f}] {r['text'][:100]}")

Stored 20 chunks in test collection
[Page 6, distance 0.9422] 2. Supervised Learning. . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 
[Page 5, distance 1.1302] Table of Contents Preface. . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . .
[Page 6, distance 1.2229]  Scaling 132 Different Kinds of Preprocessing 133 Applying Data Transformations 134 Scaling Training
